# Territory Coverage Capstone — UK & Ireland Enterprise Sales

**Author:** Maria Cordova  
**Dataset:** territory-coverage-capstone  
**Region:** UK & Ireland  
**Context:** Enterprise / Named Account Sales Coverage

---

## Business Problem

Enterprise sales organisations depend on clean territory coverage to assign accounts correctly, protect active bids, and set fair quotas. When territory data becomes messy — through duplicate ownership, wrong segment assignments, orphaned accounts, or closed entities still marked active — the business risks losing revenue, misaligning resources, and making poor quota decisions.

This project simulates a real-world territory coverage audit for a UK & Ireland enterprise sales team operating across four verticals:
- **FRIS** (Financial Services)
- **Education**
- **Public Sector**
- **Healthcare**

## What This Analysis Answers

1. What is the scale and type of territory misalignment across the portfolio?
2. How much Total Addressable Market (TAM) is at risk due to alignment issues?
3. Where are active bids most exposed?
4. What does the before vs after TAM and quota picture look like per rep?
5. What are the operational bottlenecks in the correction workflow?

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Consistent colour palette ──────────────────────────────────────────
NAVY    = '#1F4E79'
BLUE    = '#2E75B6'
LBLUE   = '#9DC3E6'
GREEN   = '#70AD47'
AMBER   = '#FFC000'
RED     = '#C00000'
LGREY   = '#F2F2F2'

VERT_COLOURS = {
    'FRIS':          NAVY,
    'Education':     BLUE,
    'Public Sector': GREEN,
    'Healthcare':    AMBER
}

plt.rcParams.update({
    'font.family':    'DejaVu Sans',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
    'axes.labelsize':     10,
    'figure.facecolor':   'white',
})

print('Libraries loaded.')

In [ ]:
# Load all five tables
acc  = pd.read_csv('/kaggle/input/territory-coverage-capstone/accounts.csv')
ter  = pd.read_csv('/kaggle/input/territory-coverage-capstone/territories.csv')
reps = pd.read_csv('/kaggle/input/territory-coverage-capstone/reps.csv')
al   = pd.read_csv('/kaggle/input/territory-coverage-capstone/account_alignment.csv')
cas  = pd.read_csv('/kaggle/input/territory-coverage-capstone/cases.csv')

# Parse dates
al['implementation_date'] = pd.to_datetime(al['implementation_date'], errors='coerce')
cas['date_opened']        = pd.to_datetime(cas['date_opened'], errors='coerce')
cas['date_closed']        = pd.to_datetime(cas['date_closed'], errors='coerce')

print(f'Accounts:          {len(acc):>4} rows  |  {acc.shape[1]} columns')
print(f'Territories:       {len(ter):>4} rows  |  {ter.shape[1]} columns')
print(f'Reps:              {len(reps):>4} rows  |  {reps.shape[1]} columns')
print(f'Account Alignment: {len(al):>4} rows  |  {al.shape[1]} columns')
print(f'Cases:             {len(cas):>4} rows  |  {cas.shape[1]} columns')

---
## 2. Dataset Overview

Before any analysis, we establish the baseline shape of the portfolio.

In [ ]:
total_tam     = acc['estimated_tam'].sum()
active_bids   = acc['active_bid_flag'].sum()
clean_pct     = (acc['hierarchy_status'] == 'Clean').mean() * 100
tam_at_risk   = al['tam_at_risk'].sum()
avg_days_open = cas['days_open'].mean()
tam_uplift    = reps['corrected_tam'].sum() - reps['current_tam'].sum()

print('=' * 50)
print('  PORTFOLIO SNAPSHOT')
print('=' * 50)
print(f'  Total accounts:          {len(acc):>6,}')
print(f'  Total TAM:               {total_tam:>10,.0f}')
print(f'  Active bid accounts:     {active_bids:>6,}  ({active_bids/len(acc)*100:.1f}%)')
print(f'  Clean hierarchy:         {clean_pct:>9.1f}%')
print(f'  Alignment issues raised: {len(al):>6,}')
print(f'  TAM at risk:             {tam_at_risk:>10,.0f}  ({tam_at_risk/total_tam*100:.1f}% of total)')
print(f'  Cases opened:            {len(cas):>6,}')
print(f'  Avg case resolution:     {avg_days_open:>9.0f} days')
print(f'  TAM uplift post-cleanup: {tam_uplift:>10,.0f}')
print('=' * 50)

---
## 3. Account Health — Hierarchy Status Breakdown

The `hierarchy_status` field classifies every account. Only **Clean** accounts have no structural issues. Everything else requires investigation or remediation.

In [ ]:
hier_order  = ['Clean', 'Misaligned', 'Wrong Parent', 'Duplicate', 'Orphan', 'Closed Entity']
hier_counts = acc['hierarchy_status'].value_counts().reindex(hier_order)
hier_pct    = (hier_counts / len(acc) * 100).round(1)

hier_colours = [GREEN, AMBER, BLUE, RED, NAVY, '#7F7F7F']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Account Hierarchy Status', fontsize=15, fontweight='bold', y=1.01)

# Bar chart
bars = axes[0].barh(hier_order[::-1], hier_counts[::-1], color=hier_colours[::-1], edgecolor='white', height=0.6)
for bar, val, pct in zip(bars, hier_counts[::-1], hier_pct[::-1]):
    axes[0].text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                 f'{val}  ({pct}%)', va='center', fontsize=10)
axes[0].set_xlabel('Number of Accounts')
axes[0].set_title('Count by Status')
axes[0].set_xlim(0, max(hier_counts) * 1.3)

# Donut — Clean vs Not Clean
clean_count    = hier_counts['Clean']
problem_count  = len(acc) - clean_count
wedge_sizes    = [clean_count, problem_count]
wedge_labels   = [f'Clean\n{clean_count} ({clean_count/len(acc)*100:.0f}%)',
                  f'Issues\n{problem_count} ({problem_count/len(acc)*100:.0f}%)']
wedge_colours  = [GREEN, RED]
wedges, texts  = axes[1].pie(wedge_sizes, labels=wedge_labels, colors=wedge_colours,
                              startangle=90, wedgeprops={'width': 0.5, 'edgecolor': 'white', 'linewidth': 2})
for t in texts:
    t.set_fontsize(11)
axes[1].set_title('Clean vs Problem Accounts')

plt.tight_layout()
plt.show()

print(f'\n{problem_count} of {len(acc)} accounts ({problem_count/len(acc)*100:.1f}%) have a hierarchy issue requiring action.')

---
## 4. TAM Distribution by Vertical

Understanding where TAM sits by vertical tells us which teams have the most revenue exposure.

In [ ]:
vert_tam = acc.groupby('vertical').agg(
    account_count=('account_id', 'count'),
    total_tam=('estimated_tam', 'sum'),
    avg_tam=('estimated_tam', 'mean'),
    active_bids=('active_bid_flag', 'sum')
).reset_index().sort_values('total_tam', ascending=False)

vert_tam['bid_pct'] = (vert_tam['active_bids'] / vert_tam['account_count'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('TAM by Vertical', fontsize=15, fontweight='bold', y=1.01)

colours = [VERT_COLOURS[v] for v in vert_tam['vertical']]

# Total TAM bars
bars = axes[0].bar(vert_tam['vertical'], vert_tam['total_tam'] / 1000,
                   color=colours, edgecolor='white', width=0.5)
for bar, row in zip(bars, vert_tam.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{row.total_tam/1000:,.0f}k', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Total TAM (000s)')
axes[0].set_title('Total TAM per Vertical')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}k'))

# Avg TAM per account
bars2 = axes[1].bar(vert_tam['vertical'], vert_tam['avg_tam'],
                    color=colours, edgecolor='white', width=0.5)
for bar, row in zip(bars2, vert_tam.itertuples()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{row.avg_tam:,.0f}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Average TAM per Account')
axes[1].set_title('Average TAM per Account by Vertical')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

print('\nVertical Summary:')
print(vert_tam[['vertical','account_count','total_tam','avg_tam','active_bids','bid_pct']]
      .to_string(index=False))

---
## 5. Alignment Issues — Scale and Severity

The `account_alignment` table captures every identified misalignment. Here we analyse what kinds of issues exist, how severe they are, and what TAM they put at risk.

In [ ]:
issue_summary = al.groupby('issue_type').agg(
    count=('alignment_id', 'count'),
    tam_at_risk=('tam_at_risk', 'sum'),
    high_sev=('issue_severity', lambda x: (x == 'High').sum())
).reset_index().sort_values('tam_at_risk', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Alignment Issues — Volume and TAM at Risk', fontsize=15, fontweight='bold', y=1.01)

# Issue count
issue_colours = [RED if row.high_sev / row.count > 0.4 else AMBER if row.high_sev > 0 else BLUE
                 for row in issue_summary.itertuples()]

bars = axes[0].barh(issue_summary['issue_type'], issue_summary['count'],
                    color=issue_colours, edgecolor='white', height=0.6)
for bar, val in zip(bars, issue_summary['count']):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=10)
axes[0].set_xlabel('Number of Issues')
axes[0].set_title('Issue Count by Type')
axes[0].set_xlim(0, issue_summary['count'].max() * 1.2)

# TAM at risk
bars2 = axes[1].barh(issue_summary['issue_type'], issue_summary['tam_at_risk'] / 1000,
                     color=issue_colours, edgecolor='white', height=0.6)
for bar, val in zip(bars2, issue_summary['tam_at_risk']):
    axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{val/1000:,.0f}k', va='center', fontsize=10)
axes[1].set_xlabel('TAM at Risk (000s)')
axes[1].set_title('TAM at Risk by Issue Type')
axes[1].set_xlim(0, issue_summary['tam_at_risk'].max() / 1000 * 1.25)

plt.tight_layout()
plt.show()

print(f'\nTotal TAM at risk across all issues: {al["tam_at_risk"].sum():,.0f}')
print(f'This represents {al["tam_at_risk"].sum() / acc["estimated_tam"].sum() * 100:.1f}% of total portfolio TAM')

---
## 6. Active Bid Risk Analysis

Active bids are the highest-stakes accounts. If an active bid is assigned to the wrong territory or rep, the business risks losing a live deal. This section isolates those accounts.

In [ ]:
# Merge alignment with account data to find active bid misalignments
al_with_acc = al.merge(acc[['account_id','active_bid_flag','vertical','estimated_tam','hierarchy_status']],
                       on='account_id', how='left')

bid_issues = al_with_acc[al_with_acc['active_bid_flag'] == 1].copy()

print(f'Total alignment issues involving active bid accounts: {len(bid_issues)}')
print(f'Of which High severity: {(bid_issues["issue_severity"] == "High").sum()}')
print(f'TAM at risk on active bids: {bid_issues["tam_at_risk"].sum():,.0f}')
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Active Bid Risk in Alignment Issues', fontsize=15, fontweight='bold', y=1.01)

# Active bid issues by severity
sev_counts = bid_issues['issue_severity'].value_counts().reindex(['High','Medium','Low'])
sev_cols   = [RED, AMBER, GREEN]
axes[0].bar(sev_counts.index, sev_counts.values, color=sev_cols, edgecolor='white', width=0.5)
for i, (idx, val) in enumerate(sev_counts.items()):
    axes[0].text(i, val + 0.3, str(val), ha='center', fontweight='bold')
axes[0].set_title('Active Bid Issues by Severity')
axes[0].set_ylabel('Count')

# Active bid issues by vertical
bid_vert = bid_issues.groupby('vertical')['alignment_id'].count().sort_values(ascending=False)
colours  = [VERT_COLOURS.get(v, BLUE) for v in bid_vert.index]
axes[1].bar(bid_vert.index, bid_vert.values, color=colours, edgecolor='white', width=0.5)
for i, val in enumerate(bid_vert.values):
    axes[1].text(i, val + 0.2, str(val), ha='center', fontweight='bold')
axes[1].set_title('Active Bid Issues by Vertical')
axes[1].set_ylabel('Count')

# Resolution status of active bid issues
res_counts = bid_issues['resolution_status'].value_counts()
res_cols   = {'Implemented': GREEN, 'In Progress': BLUE, 'Open': AMBER, 'Rejected': RED}
colours_r  = [res_cols.get(r, LBLUE) for r in res_counts.index]
axes[2].pie(res_counts.values, labels=res_counts.index, colors=colours_r,
            autopct='%1.0f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[2].set_title('Resolution Status of Active Bid Issues')

plt.tight_layout()
plt.show()

---
## 7. Before vs After — Rep TAM and Quota Impact

The `reps` table captures each rep's current and corrected TAM and quota. This is the core commercial story: what does cleanup actually deliver for the business?

In [ ]:
reps_sorted = reps.sort_values('corrected_tam', ascending=True).copy()
reps_sorted['tam_uplift']   = reps_sorted['corrected_tam']   - reps_sorted['current_tam']
reps_sorted['quota_uplift'] = reps_sorted['corrected_quota'] - reps_sorted['current_quota']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Rep TAM and Quota — Before vs After Cleanup', fontsize=15, fontweight='bold', y=1.01)

y = np.arange(len(reps_sorted))
bar_h = 0.35

# TAM before/after
axes[0].barh(y - bar_h/2, reps_sorted['current_tam']   / 1000, bar_h,
             label='Current TAM',   color=LBLUE, edgecolor='white')
axes[0].barh(y + bar_h/2, reps_sorted['corrected_tam'] / 1000, bar_h,
             label='Corrected TAM', color=NAVY, edgecolor='white')
axes[0].set_yticks(y)
axes[0].set_yticklabels(reps_sorted['rep_name'], fontsize=9)
axes[0].set_xlabel('TAM (000s)')
axes[0].set_title('TAM Before vs After')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}k'))

# Quota before/after
axes[1].barh(y - bar_h/2, reps_sorted['current_quota']   / 1000, bar_h,
             label='Current Quota',   color=LBLUE, edgecolor='white')
axes[1].barh(y + bar_h/2, reps_sorted['corrected_quota'] / 1000, bar_h,
             label='Corrected Quota', color=NAVY, edgecolor='white')
axes[1].set_yticks(y)
axes[1].set_yticklabels(reps_sorted['rep_name'], fontsize=9)
axes[1].set_xlabel('Quota (000s)')
axes[1].set_title('Quota Before vs After')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}k'))

plt.tight_layout()
plt.show()

print(f'Total TAM uplift across all reps:   {reps_sorted["tam_uplift"].sum():>10,.0f}')
print(f'Total Quota uplift across all reps: {reps_sorted["quota_uplift"].sum():>10,.0f}')
print(f'\nTop 3 reps by TAM uplift:')
print(reps_sorted.nlargest(3, 'tam_uplift')[['rep_name','current_tam','corrected_tam','tam_uplift']].to_string(index=False))

---
## 8. Territory Health and Optimisation

The `st_status` field flags territories needing structural review. Over-fragmented territories are split too thin. Territories needing consolidation have too many overlapping issues.

In [ ]:
st_summary = ter.groupby('st_status').agg(
    count=('territory_id', 'count'),
    avg_tam=('current_tam', 'mean'),
    avg_quota=('quota', 'mean'),
    avg_accounts=('current_account_count', 'mean')
).reset_index().sort_values('avg_tam', ascending=False)

st_colours = {'Active': GREEN, 'Clean': BLUE, 'Needs Consolidation': AMBER, 'Over-fragmented': RED}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Territory Health by ST Status', fontsize=15, fontweight='bold', y=1.01)

colours = [st_colours.get(s, BLUE) for s in st_summary['st_status']]

# Territory count by status
axes[0].bar(st_summary['st_status'], st_summary['count'], color=colours, edgecolor='white', width=0.5)
for i, val in enumerate(st_summary['count']):
    axes[0].text(i, val + 0.1, str(val), ha='center', fontweight='bold')
axes[0].set_title('Territory Count by Status')
axes[0].set_ylabel('Count')

# Average TAM by status
axes[1].bar(st_summary['st_status'], st_summary['avg_tam'] / 1000,
            color=colours, edgecolor='white', width=0.5)
for i, val in enumerate(st_summary['avg_tam']):
    axes[1].text(i, val/1000 + 0.5, f'{val/1000:,.0f}k', ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Average TAM per Territory by Status')
axes[1].set_ylabel('Average TAM (000s)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}k'))

plt.tight_layout()
plt.show()

print('Territory Status Summary:')
print(st_summary[['st_status','count','avg_tam','avg_quota','avg_accounts']]
      .rename(columns={'avg_tam':'Avg TAM','avg_quota':'Avg Quota','avg_accounts':'Avg Accounts'})
      .to_string(index=False))

---
## 9. Case Workload and Operational Bottlenecks

The `cases` table tracks the correction workflow. A high volume of cases stuck in `Submitted` or `Delayed` status signals a bottleneck in the process.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Case Workload and Operational Bottlenecks', fontsize=15, fontweight='bold', y=1.01)

# Cases by owning team
team_counts = cas['owning_team'].value_counts()
team_cols   = [NAVY, BLUE, GREEN, AMBER]
axes[0].bar(team_counts.index, team_counts.values, color=team_cols, edgecolor='white', width=0.5)
for i, val in enumerate(team_counts.values):
    axes[0].text(i, val + 0.5, str(val), ha='center', fontweight='bold')
axes[0].set_title('Cases by Owning Team')
axes[0].set_ylabel('Count')

# Cases by implementation status
status_order  = ['Submitted', 'Approved', 'Implemented', 'Closed', 'Delayed']
status_counts = cas['implementation_status'].value_counts().reindex(status_order)
status_cols   = [AMBER, BLUE, GREEN, NAVY, RED]
axes[1].bar(status_counts.index, status_counts.values, color=status_cols, edgecolor='white', width=0.5)
for i, val in enumerate(status_counts.values):
    axes[1].text(i, val + 0.3, str(val), ha='center', fontweight='bold')
axes[1].set_title('Cases by Implementation Status')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)

# Average days open by team
avg_days = cas.groupby('owning_team')['days_open'].mean().sort_values(ascending=False)
colours  = [NAVY, BLUE, GREEN, AMBER]
axes[2].barh(avg_days.index, avg_days.values, color=colours, edgecolor='white', height=0.5)
for i, val in enumerate(avg_days.values):
    axes[2].text(val + 1, i, f'{val:.0f} days', va='center', fontsize=10)
axes[2].set_title('Avg Days Open by Team')
axes[2].set_xlabel('Days')
axes[2].set_xlim(0, avg_days.max() * 1.2)

plt.tight_layout()
plt.show()

open_cases    = cas[cas['implementation_status'].isin(['Submitted','Delayed'])]
closed_cases  = cas[cas['implementation_status'].isin(['Implemented','Closed'])]
print(f'Open / stalled cases:     {len(open_cases)} ({len(open_cases)/len(cas)*100:.1f}%)')
print(f'Completed cases:          {len(closed_cases)} ({len(closed_cases)/len(cas)*100:.1f}%)')
print(f'Avg days to close:        {closed_cases["days_open"].mean():.0f} days')

---
## 10. Resolution Progress Over Time

Tracking when alignment issues were implemented shows whether the cleanup programme accelerated or stalled over time.

In [ ]:
implemented = al[al['resolution_status'] == 'Implemented'].copy()
implemented['month'] = implemented['implementation_date'].dt.to_period('M')
monthly = implemented.groupby('month').agg(
    issues_fixed=('alignment_id', 'count'),
    tam_recovered=('tam_at_risk', 'sum')
).reset_index()
monthly['month_str']   = monthly['month'].astype(str)
monthly['cumulative_tam'] = monthly['tam_recovered'].cumsum()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Alignment Issue Resolution Over Time', fontsize=15, fontweight='bold', y=1.01)

# Monthly issues fixed
axes[0].bar(monthly['month_str'], monthly['issues_fixed'], color=BLUE, edgecolor='white', width=0.6)
axes[0].set_title('Issues Fixed per Month')
axes[0].set_ylabel('Issues Resolved')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(monthly['issues_fixed'].mean(), color=RED, linestyle='--', linewidth=1.5,
                label=f'Monthly avg: {monthly["issues_fixed"].mean():.1f}')
axes[0].legend()

# Cumulative TAM recovered
axes[1].fill_between(monthly['month_str'], monthly['cumulative_tam'] / 1000,
                     alpha=0.3, color=GREEN)
axes[1].plot(monthly['month_str'], monthly['cumulative_tam'] / 1000,
             color=GREEN, linewidth=2.5, marker='o', markersize=5)
axes[1].set_title('Cumulative TAM Recovered (000s)')
axes[1].set_ylabel('Cumulative TAM (000s)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}k'))

plt.tight_layout()
plt.show()

print(f'Implementation period: {monthly["month_str"].iloc[0]} to {monthly["month_str"].iloc[-1]}')
print(f'Total issues resolved: {monthly["issues_fixed"].sum()}')
print(f'Total TAM recovered:   {monthly["tam_recovered"].sum():,.0f}')

---
## 11. Key Findings and Recommendations

### What the data shows

**1. Scale of the problem**  
57.8% of accounts had a hierarchy issue at baseline. The most common problems were Misaligned (90 accounts), Wrong Parent (40), and Duplicate (40). These are not edge cases — they represent a systemic data quality problem that affects revenue planning.

**2. TAM at risk is material**  
£3.7M of TAM sat behind misaligned accounts — 56% of the total portfolio. Duplicate Ownership and Wrong Segment accounted for the largest share. Without cleanup, quota setting and territory sizing were both distorted.

**3. Active bids were exposed**  
118 accounts had active bids. 80 of those appeared in alignment issues, with 30 rated High severity. A live deal assigned to the wrong rep or territory is a direct commercial risk — not just a data hygiene issue.

**4. Cleanup delivered measurable uplift**  
Post-correction TAM across the rep team increased by £354,000. Corrected quota increased by £80,000. Daniel Brooks and Laura Bennett saw the largest individual uplifts, consistent with their territories carrying the most alignment issues.

**5. Operational bottlenecks exist**  
37% of cases were still in Submitted or Delayed status. MDM carried the highest average days open. This suggests the correction workflow itself needs process improvement — not just the data.

---

### Recommendations

| Priority | Action | Owner |
|----------|--------|-------|
| High | Resolve all 25 Active Bid Misaligned issues immediately | Sales Ops + MDM |
| High | Audit all 40 Duplicate accounts for consolidation or removal | MDM |
| Medium | Review 40 Wrong Parent accounts and correct hierarchy | BA |
| Medium | Consolidate Over-fragmented territories (4 identified) | RevOps |
| Low | Establish monthly data quality review cadence | Sales Ops |

---

### Skills demonstrated in this project

- Relational data modelling across 5 connected tables
- Data cleaning and integrity validation
- Exploratory data analysis with pandas
- Business-context visualisation with matplotlib
- Translating data findings into commercial recommendations